# Gemini로 네이버 뉴스 요약 및 분석 챗봇 만들기
[https://ai.google.dev/gemini-api/docs?hl=ko&authuser=1]([https://ai.google.dev/gemini-api/docs?hl=ko&authuser=1])

In [4]:
import os 
import gradio as gr
from google import genai
from dotenv import load_dotenv
load_dotenv("./.gemini_env")
api_key = os.getenv("google_api_key")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="인공지능과 개발자의 미래에 대해 알려줘",
)

print(response.text)

인공지능(AI)의 발전은 개발자라는 직업의 정의와 업무 방식을 근본적으로 바꾸고 있습니다. 많은 이들이 "AI가 개발자를 대체할 것인가?"를 걱정하지만, 전문가들의 공통된 의견은 **"AI가 개발자를 대체하는 것이 아니라, AI를 사용하는 개발자가 그렇지 못한 개발자를 대체할 것"**이라는 점입니다.

인공지능과 개발자의 미래를 4가지 핵심 관점으로 정리해 드립니다.

---

### 1. 개발 방식의 패러다임 변화 (How we work)
과거에는 문법을 외우고 코드를 직접 한 줄씩 치는 '코딩' 능력이 중요했다면, 미래에는 **AI와 협업하는 능력**이 핵심이 됩니다.

*   **생산성 극대화:** GitHub Copilot, ChatGPT 같은 도구들이 단순 반복적인 코드(Boilerplate code) 작성, 단위 테스트 생성, 버그 찾기 등을 대신해 줍니다. 이로 인해 개발 속도가 비약적으로 빨라집니다.
*   **언어의 장벽 완화:** 특정 프로그래밍 언어의 문법을 완벽히 몰라도, 논리적 흐름만 알면 AI의 도움을 받아 다른 언어로 쉽게 전환하거나 구현할 수 있게 됩니다.
*   **추상화 수준의 상승:** 개발자는 '어떻게 구현할 것인가(How)'보다 '무엇을 만들 것인가(What)'와 '왜 만드는가(Why)'에 더 집중하게 됩니다.

### 2. 요구되는 역량의 변화 (New Skillsets)
단순 구현 기술보다는 상위 수준의 설계 능력이 더욱 중요해집니다.

*   **문제 정의 및 기획 능력:** 사용자의 요구사항을 정확히 파악하고 이를 기술적인 명세로 전환하는 능력이 중요해집니다.
*   **시스템 아키텍처 및 설계:** 전체적인 서비스의 구조를 짜고, 다양한 기술 스택을 어떻게 연결할지 결정하는 '설계자'로서의 역량이 강조됩니다.
*   **코드 리뷰 및 검증 능력:** AI가 짠 코드가 정확한지, 보안상 취약점은 없는지, 효율적인지를 판단하는 '검토자'의 역할이 커집니다. (AI의 할루시네이션(환각) 현상 때문)
*   **커뮤니케이

# UI를 붙여서 (gradio 활용해서) 챗봇 만들기

In [6]:
import os 
import gradio as gr
from google import genai
from dotenv import load_dotenv
load_dotenv("./.gemini_env")
api_key = os.getenv("google_api_key")

client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3-flash-preview"


def chat_with_gemini(message, history, chat_session):
    try:
        # 세션이 없으면 새로 생성
        if chat_session is None:
            chat_session = client.chats.create(model=MODEL_NAME)

        response = chat_session.send_message(message)
        answer = response.text if response.text else "응답을 받지 못했습니다."

        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": answer})

        return history, chat_session, ""

    except Exception as e:
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": f"오류가 발생했습니다: {str(e)}"})
        return history, chat_session, ""


def reset_chat():
    return [], None, ""


with gr.Blocks(title="Gemini Gradio Chatbot") as demo:
    gr.Markdown("# Gemini + Gradio 챗봇")
    gr.Markdown("Google GenAI SDK와 Gradio로 만든 간단한 챗봇입니다.")

    chatbot = gr.Chatbot(type="messages", height=500)
    msg = gr.Textbox(
        placeholder="메시지를 입력하세요...",
        label="질문",
        lines=1
    )

    chat_state = gr.State(None)

    with gr.Row():
        send_btn = gr.Button("전송")
        clear_btn = gr.Button("대화 초기화")

    send_btn.click(
        fn=chat_with_gemini,
        inputs=[msg, chatbot, chat_state],
        outputs=[chatbot, chat_state, msg]
    )

    msg.submit(
        fn=chat_with_gemini,
        inputs=[msg, chatbot, chat_state],
        outputs=[chatbot, chat_state, msg]
    )

    clear_btn.click(
        fn=reset_chat,
        inputs=[],
        outputs=[chatbot, chat_state, msg]
    )

demo.launch(inline=False, share=False)

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [7]:
demo.close()

Closing server running on port: 7861


# 네이버에서 뉴스 제목 수집 후 gemini로 내용 요약하기
* 1. 네이버에서 뉴스 수집
* 2. 제목 가져오기
* 3. 제목 선택
* 4. 기사 내용 가져오기
* 5. 기사 내용 gemini에게 요약시키기

# 네이버 API에서 뉴스 수집

In [44]:
import os 
import re
import requests
import pandas as pd
import gradio as gr
from google import genai
from bs4 import BeautifulSoup as bs
from dotenv import load_dotenv
load_dotenv("./.gemini_env")


def text_clean(text):
    temp = re.sub(r"</?[^>]+>", "", text)
    temp = re.sub("[^가-힣a-zA-Z09]", " ", temp)
    temp = temp.replace("\n", " ").replace("\t", " ").replace("  ", " ").replace("  ", " ")
    temp = temp.replace("  ", " ").strip()
    return temp

# 네이버 뉴스 검색 함수
def search_news(keyword):
#     keyword = "삼성전자"
    url = "https://openapi.naver.com/v1/search/news"
    payload = dict(query=keyword, display=100, start=1, sort='date')
    headers = {"X-Naver-Client-Id" : os.getenv("Client_ID"), 
               "X-Naver-Client-Secret": os.getenv("Client_Secret")}

    r = requests.get(url, params=payload, headers=headers)
    print(r.url)
    print(r.status_code)
    data = r.json()

    result = {}
    for item in data['items']:
        for key, value in item.items():
            if key in ('title', "description"):
                value=text_clean(value)
            result.setdefault(key,[]).append(value)

    df = pd.DataFrame(result)     
    return df[['title', 'originallink']].head(10)


def content_extract(news_10):
    full_text = ""
    for link in news_10['originallink']:
        try:
            r = requests.get(link)
        except:
            continue
        soup = bs(r.text, "lxml")
        paragraphs = soup.select('[id*="content"], [class*="content"]')    
        
        for tags in paragraphs:
            if len(tags.get_text()) > 20:
                full_text += text_clean(tags.get_text())
    return full_text

def summary_gemini(full_text):
    prompt = f"""다음 기사를 주제별로 분류해서 한국어로 주제별 500자 이내로 요약해줘.
                 그리고 이 기사 내용이 핀테크 분야와 경제 상황에 미칠 영향을 자세히 분석해줘:\n\n
                 {full_text}
                 """
    prompt

    api_key = os.getenv("google_api_key")

    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
    )

    return response.text

keyword = input()
news_10 = search_news(keyword)
full_text = content_extract(news_10)
final_result = summary_gemini(full_text)
print(final_result)

소상공인 마이데이터
https://openapi.naver.com/v1/search/news?query=%EC%86%8C%EC%83%81%EA%B3%B5%EC%9D%B8+%EB%A7%88%EC%9D%B4%EB%8D%B0%EC%9D%B4%ED%84%B0&display=100&start=1&sort=date
200
제공해주신 기사 내용을 바탕으로 주제별 요약과 핀테크 및 경제 상황에 미칠 영향 분석을 정리해 드립니다.

---

### 1. 주제별 요약 (각 500자 이내)

#### **① 소상공인·자영업자 맞춤형 행정 및 금융 지원**
중소벤처기업부는 소상공인의 업종, 지역, 과거 지원 이력 등 약 18만 건의 데이터를 분석해 맞춤형 지원 정보를 카카오톡으로 안내하는 ‘소상공인 알림톡’ 서비스를 시작했습니다. 이 서비스는 정보 부족으로 신청 시기를 놓치는 자영업자들을 위해 실시간으로 적합한 지원사업만 골라 안내하며, 중복 안내를 방지하는 기능도 갖췄습니다. 또한 제주도는 소상공인 지원사업 신청을 온라인화하여 서류 없는 행정을 구현하는 시범 사업을 추진하고 있습니다.

#### **② 금융권의 ‘생산적·포용적 금융’ 확대와 ESG 경영**
KB금융그룹은 10조 원 규모의 ‘국민성장 프로젝트’를 통해 첨단전략산업 육성과 소상공인 지원에 나섰습니다. 이와 더불어 금융권 전반에서 사회공헌 활동이 활발합니다. 새마을금고는 독거노인에게 AI 반려로봇을 지원하고, 교보생명은 장애 학생 대상 금융 교육을, KB국민카드는 발달장애 청소년 미술 전시를 개최했습니다. 이는 단순 대출 지원을 넘어 금융의 사회적 책임을 다하고 취약계층의 금융 접근성을 높이는 포용적 금융으로 진화하고 있음을 보여줍니다.

#### **③ AI 및 디지털 기술을 활용한 자산관리·플랫폼 혁신**
하나은행은 AI가 투자 성향과 목표를 진단해 맞춤형 포트폴리오를 제공하는 ‘AI 연금투자 솔루션’을 통합 서비스로 선보였습니다. 빗썸은 가독성을 높인 전용 서체 도입과 UX 개편을 통해 매매 환경을 정교화했으며,

# GPT한테 gradio로 chatbot UI 씌워달라고 부탁한 후 코드 받아서 실행하기

In [3]:
# !pip install gradio

In [2]:
import os
import re
import requests
import pandas as pd
import gradio as gr

from dotenv import load_dotenv
from google import genai
from bs4 import BeautifulSoup as bs

load_dotenv("./.gemini_env")


# ---------------------------
# 텍스트 정리 함수
# ---------------------------
def text_clean(text):
    temp = re.sub(r"</?[^>]+>", "", str(text))
    temp = re.sub(r"[^가-힣a-zA-Z0-9]", " ", temp)
    temp = re.sub(r"\s+", " ", temp).strip()
    return temp


# ---------------------------
# 네이버 뉴스 검색 함수
# ---------------------------
def search_news(keyword):
    url = "https://openapi.naver.com/v1/search/news"
    payload = {
        "query": keyword,
        "display": 10,   # 우선 10개만
        "start": 1,
        "sort": "date"
    }
    headers = {
        "X-Naver-Client-Id": os.getenv("Client_ID"),
        "X-Naver-Client-Secret": os.getenv("Client_Secret")
    }

    r = requests.get(url, params=payload, headers=headers, timeout=10)
    r.raise_for_status()

    data = r.json()
    items = data.get("items", [])

    result = {}
    for item in items:
        for key, value in item.items():
            if key in ("title", "description"):
                value = text_clean(value)
            result.setdefault(key, []).append(value)

    if not result:
        return pd.DataFrame(columns=["title", "originallink"])

    df = pd.DataFrame(result)

    needed_cols = [col for col in ["title", "originallink"] if col in df.columns]
    return df[needed_cols].head(10)


# ---------------------------
# 뉴스 본문 추출 함수
# ---------------------------
def content_extract(news_df):
    full_text = ""
    collected_links = []

    if news_df is None or news_df.empty:
        return full_text, collected_links

    for link in news_df["originallink"]:
        try:
            r = requests.get(link, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
        except Exception:
            continue

        soup = bs(r.text, "lxml")

        # id나 class에 content가 포함된 태그 전부 탐색
        paragraphs = soup.select('[id*="content"], [class*="content"]')

        page_text = []
        for tag in paragraphs:
            txt = text_clean(tag.get_text(" ", strip=True))
            if len(txt) > 20:
                page_text.append(txt)

        if page_text:
            full_text += " ".join(page_text) + "\n"
            collected_links.append(link)

    return full_text.strip(), collected_links


# ---------------------------
# Gemini 요약 함수
# ---------------------------
def summary_gemini(full_text):
    if not full_text.strip():
        return "본문을 추출하지 못했습니다. 다른 키워드로 다시 시도해 주세요."

    prompt = f"""
다음 뉴스 기사들을 주제별로 분류해서 한국어로 정리해줘.

요구사항:
1. 주제별로 묶어라.
2. 각 주제 요약은 500자 이내로 작성해라.
3. 마지막에는 '핀테크 분야 영향'과 '경제 전반 영향'을 구분해서 자세히 분석해라.
4. 전체 답변은 보기 쉽게 제목과 항목을 나눠서 작성해라.

기사 원문:
{full_text}
"""

    api_key = os.getenv("google_api_key")
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text


# ---------------------------
# 챗봇 응답 함수
# ---------------------------
def chatbot_response(message, history):
    keyword = message.strip()

    if not keyword:
        return "검색할 키워드를 입력해 주세요."

    try:
        news_df = search_news(keyword)

        if news_df.empty:
            return f"'{keyword}'에 대한 뉴스 검색 결과가 없습니다."

        full_text, used_links = content_extract(news_df)

        if not full_text:
            return (
                f"'{keyword}' 뉴스는 찾았지만 본문 추출에 실패했어요.\n\n"
                "가능한 원인:\n"
                "- 언론사 페이지 구조가 달라서 본문 선택이 안 됨\n"
                "- 접속 차단 또는 동적 렌더링 페이지\n"
            )

        summary = summary_gemini(full_text)

        news_list_text = "\n".join(
            [f"{i+1}. {title}" for i, title in enumerate(news_df["title"].tolist())]
        )

        link_text = "\n".join([f"- {link}" for link in used_links[:10]])

        final_answer = f"""검색 키워드: {keyword}

[검색된 뉴스 제목]
{news_list_text}

[요약 및 분석]
{summary}

[본문 추출에 사용된 링크]
{link_text}
"""
        return final_answer

    except Exception as e:
        return f"오류가 발생했습니다: {type(e).__name__}: {e}"


# ---------------------------
# 예시 입력
# ---------------------------
examples = [
    ["삼성전자"],
    ["카카오페이"],
    ["비트코인"],
    ["금리"],
    ["핀테크"]
]


# ---------------------------
# Gradio UI
# ---------------------------
with gr.Blocks(title="뉴스 요약 챗봇") as demo:
    gr.Markdown(
        """
# 뉴스 요약 챗봇
네이버 뉴스 검색 결과를 모아서 기사 본문을 추출한 뒤,
Gemini로 주제별 요약과 핀테크/경제 영향 분석을 제공합니다.
"""
    )

    chatbot = gr.Chatbot(height=500, type="messages")
    msg = gr.Textbox(
        label="검색 키워드 입력",
        placeholder="예: 삼성전자, 비트코인, 금리, 핀테크"
    )
    clear_btn = gr.Button("대화 초기화")

    gr.Examples(
        examples=examples,
        inputs=msg
    )

    def user_submit(user_message, history):
        history = history or []
        history.append({"role": "user", "content": user_message})
        return "", history

    def bot_submit(history):
        user_message = history[-1]["content"]
        bot_message = chatbot_response(user_message, history)
        history.append({"role": "assistant", "content": bot_message})
        return history

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        bot_submit, chatbot, chatbot
    )

    clear_btn.click(lambda: [], None, chatbot, queue=False)
    
demo.launch(inline=False, share=False)

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [1]:
demo.close() # 무조건 해줘야 함

NameError: name 'demo' is not defined